In [152]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

In [80]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')


In [81]:
df_train['train'] = 1
df_test['train'] = 0

In [82]:
df_com = pd.concat([df_train, df_test], axis=0, ignore_index=True)

In [83]:
df_com

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,1305,NaN,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S,0
1305,1306,NaN,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C,0
1306,1307,NaN,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S,0
1307,1308,NaN,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S,0


In [84]:
df_com.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   object 
 4   Sex          1309 non-null   object 
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   object 
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    object 
 11  Embarked     1307 non-null   object 
 12  train        1309 non-null   int64  
dtypes: float64(3), int64(5), object(5)
memory usage: 133.1+ KB


In [85]:
# fill age
df_com['Age'] = df_com.groupby('Sex')['Age'].transform(lambda x:x.fillna(x.median()))

In [86]:
df_com['Fare'] = df_com['Fare'].fillna(df_com['Fare'].mean())

In [87]:
df_com['Embarked'] = df_com['Embarked'].fillna(df_com['Embarked'].mode()[0])

In [88]:
df_com['Title'] = df_com['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

In [89]:
df_com['Family_size'] = df_com['SibSp'] + df_com['Parch']

In [90]:
df_com['FareBin'] = pd.qcut(df_com['Fare'], 5)
df_com['FareBin_Code'] = df_com['FareBin'].cat.codes

In [91]:
df_com['AgeBin'] = pd.qcut(df_com['Age'], 4)
df_com['AgeBin_Code'] = df_com['AgeBin'].cat.codes

In [92]:
df_com['AgeBin_Code']

0       0
1       3
2       1
3       2
4       2
       ..
1304    1
1305    3
1306    3
1307    1
1308    1
Name: AgeBin_Code, Length: 1309, dtype: int8

In [95]:
df_fea = df_com.drop(columns=['PassengerId', 'AgeBin', 'FareBin', 'Cabin', 'Ticket', 'Name'])

In [104]:
df_fea['Title'].unique()

array(['Mr', 'Mrs', 'Miss', 'Master', 'Don', 'Rev', 'Dr', 'Mme', 'Ms',
       'Major', 'Lady', 'Sir', 'Mlle', 'Col', 'Capt', 'Countess',
       'Jonkheer', 'Dona'], dtype=object)

In [106]:
df_fea['Title'] = df_fea['Title'].replace({
    'Master': 'Mr',
    'Don' : 'Mr',
    'Rev' : 'Mr',
    'Dr' : 'Mr',
    'Mme' : 'Mrs',
    'Ms' : 'Miss',
    'Major' : 'Mr',
    'Lady' : 'Mrs',
    'Sir' : 'Mr',
    'Mlle' : 'Miss',
    'Col' : 'Mr',
    'Capt' : 'Mr',
    'Countess' : 'Mrs',
    'Jonkheer' : 'Mr',
    'Dona' : 'Mr'
})

In [108]:
df_fea

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,train,Title,Family_size,FareBin_Code,AgeBin_Code
0,0.0,3,male,22.0,1,0,7.2500,S,1,Mr,1,0,0
1,1.0,1,female,38.0,1,0,71.2833,C,1,Mrs,1,4,3
2,1.0,3,female,26.0,0,0,7.9250,S,1,Miss,0,1,1
3,1.0,1,female,35.0,1,0,53.1000,S,1,Mrs,1,4,2
4,0.0,3,male,35.0,0,0,8.0500,S,1,Mr,0,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,NaN,3,male,28.0,0,0,8.0500,S,0,Mr,0,1,1
1305,NaN,1,female,39.0,0,0,108.9000,C,0,Mr,0,4,3
1306,NaN,3,male,38.5,0,0,7.2500,S,0,Mr,0,0,3
1307,NaN,3,male,28.0,0,0,8.0500,S,0,Mr,0,1,1


In [109]:
cat_fea = df_fea[['Sex', 'Embarked', 'Title']]

In [112]:
ohe = OneHotEncoder()
get = ohe.fit_transform(cat_fea)

In [129]:
df_fea_num = df_fea.drop(columns=cat_fea)

In [128]:
df_fea_cat = pd.DataFrame(get.toarray(), columns=ohe.get_feature_names_out())

In [136]:
df_processed = pd.concat([df_fea_num, df_fea_cat], axis = 1)

In [137]:
df_processed

,Survived,Pclass,Age,SibSp,Parch,Fare,train,Family_size,FareBin_Code,AgeBin_Code,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Title_Miss,Title_Mr,Title_Mrs
0,0.0,3,22.0,1,0,7.2500,1,1,0,0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.0,1,38.0,1,0,71.2833,1,1,4,3,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1.0,3,26.0,0,0,7.9250,1,0,1,1,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,1.0,1,35.0,1,0,53.1000,1,1,4,2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,3,35.0,0,0,8.0500,1,0,1,2,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,NaN,3,28.0,0,0,8.0500,0,0,1,1,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1305,NaN,1,39.0,0,0,108.9000,0,0,4,3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1306,NaN,3,38.5,0,0,7.2500,0,0,0,3,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1307,NaN,3,28.0,0,0,8.0500,0,0,1,1,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0


In [144]:
scaler = StandardScaler()

In [145]:
X_train = df_processed[df_processed['train'] == 1].drop(['Survived'], axis = 1)
X_train_scaled = scaler.fit_transform(X_train)

In [146]:
X_test = df_processed[df_processed['train'] == 0].drop(['Survived'], axis = 1)
X_test_scaled = scaler.transform(X_test)

In [143]:
y_train = df_processed[df_processed['train'] == 1]['Survived']

In [192]:
train_set = TensorDataset( 
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
)

test_set = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32)
)

In [271]:
torch.tensor(X_test_scaled, dtype=torch.float32).shape

torch.Size([418, 17])

In [194]:
train_loader = DataLoader(
    train_set, batch_size=64, shuffle=True
)

test_loader = DataLoader(
    test_set, batch_size=64
)

In [ ]:
class ANN(nn.Module):
    def __init__(self, input):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

In [305]:
model = ANN(X_test_scaled.shape[1])
critirion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

In [311]:
epochs = 100
best_loss = float('inf')
best_weights = None

model.train()
for epoch in range(epochs):
    for Xb, yb in train_loader:
        epoch_loss = 0.0
        optimizer.zero_grad()

        output = model(Xb.float())
        loss = critirion(output, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step(loss.item())

    if loss.item() < best_loss:
        best_loss = loss.item()
        best_weights = model.state_dict()
    print(f'for {epoch+1} loss is {epoch_loss/len(train_loader)}')

C:\Users\Tanvir\AppData\Local\Temp\ipykernel_22432\1209777778.py:16: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(loss.item())


for 1 loss is 0.020367681980133057
for 2 loss is 0.03439971165997641
for 3 loss is 0.024552719933646067
for 4 loss is 0.031056919268199375
for 5 loss is 0.037630638905933926
for 6 loss is 0.027296319603919983
for 7 loss is 0.027214350444929942
for 8 loss is 0.022338562778064182
for 9 loss is 0.026753808770860945
for 10 loss is 0.0280958605664117
for 11 loss is 0.021462325538907732
for 12 loss is 0.034892533506665914
for 13 loss is 0.02184808679989406
for 14 loss is 0.026750515614237105
for 15 loss is 0.02516533008643559
for 16 loss is 0.03489216097763607
for 17 loss is 0.02671693904059274
for 18 loss is 0.03260326598371778
for 19 loss is 0.022793527160372053
for 20 loss is 0.023315563797950745
for 21 loss is 0.024573592203004018
for 22 loss is 0.025419590728623525
for 23 loss is 0.021185606718063354
for 24 loss is 0.019912609032222202
for 25 loss is 0.025991124766213552
for 26 loss is 0.023224066410745894
for 27 loss is 0.03174393943377903
for 28 loss is 0.02691126082624708
for 29 loss

In [312]:

model.load_state_dict(best_weights)

model.eval()
preds = []
with torch.no_grad():
    for Xb in test_loader:
        # print(Xb)
        output = model(Xb[0])
        predicted = output.round()
        # if output >= 0.5:
        #     output = 1
        # else:
        #     output = 0

        preds.append(predicted)
        

In [313]:
y_pred = torch.cat(preds).numpy().flatten().astype(int)

In [314]:
df_test['Survived'] = y_pred

In [315]:
df_test[['PassengerId','Survived' ]].to_csv('output.csv' ,index=False ) 